### Включите акселератор!!! Settings > Accelerator > GPU T4 X2

In [11]:
!pip install -q "peft==0.12.0" transformers datasets accelerate bitsandbytes

In [6]:
# 2. Импорт библиотек
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

In [7]:
# 3. Загрузка модели и токенизатора
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [8]:
# ==========================================
# ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ ДЛЯ ТЕСТОВ
# ==========================================
def test_model(prompt, current_model, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt").to(current_model.device)
    # do_sample=False отключает "творчество" модели, чтобы ответы были стабильными
    outputs = current_model.generate(**inputs, max_new_tokens=60, do_sample=False)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Тестовые вопросы (их НЕТ в датасете для обучения!)
test_prompts = [
    "Вопрос: Который сейчас час? Ответ:",
    "Вопрос: Ты любишь слушать музыку? Ответ:",
    "Вопрос: Что будешь делать на выходных? Ответ:"
]

In [9]:
# ==========================================
# ЭТАП 1: ПРОВЕРКА ДО ОБУЧЕНИЯ (БАЗОВАЯ МОДЕЛЬ)
# ==========================================
print("\n" + "="*60)
print("ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")


ЭТАП 1: ОТВЕТЫ БАЗОВОЙ МОДЕЛИ (ДО ПРИМЕНЕНИЯ LoRA)
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: 12.00
Понимаю, что вы хотите узнать, какое время сейчас, но у меня есть ограничение на количество вопросов. Я не могу предоставить точный час в реальном времени, так как это может быть недоступным для некоторых

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Да, я люблю слушать музыку. Какую музыку вы предпочитаете? Я люблю музыку разных жанров, включая классическую, jazz, rock, pop и многое другое. Какой год был для тебя самым важным

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: Я не могу предсказать, что будет на выходных. Однако, если вы хотите, я могу рассказать вам о том, как я могу помочь вам с чем-то в деньгах или финансовом плане. Если у вас есть конкретные вопросы или задачи, которые



In [13]:
!pip install --upgrade "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 95.8 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [14]:
# ==========================================
# ЭТАП 2: НАСТРОЙКА И ЗАПУСК LoRA
# ==========================================
print("Настраиваем LoRA и запускаем обучение...")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, lora_config)
print("Обучаемых параметров:", end=" ")
model.print_trainable_parameters()

Настраиваем LoRA и запускаем обучение...
Обучаемых параметров: trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


### ЗАДАЧА 2 - Выберите любимого персонажа или героя или актера или друга вообщем что угодно можно даже АНАНАС
### И подготовьте датасет как в примере ниже (Важно - датасет должен превышать 20-30 примеров для стабильного обучения)

In [15]:

# Микро-датасет стиль "рыба"
data = [
    {"text": "Вопрос: Как дела? Ответ: Буль🫧."},
    {"text": "Вопрос: Что делаешь? Ответ: Буль Буль🫧."},
    {"text": "Вопрос: Будешь чай? Ответ: Буль Бу🫧."},
    {"text": "Вопрос: Какой сегодня день? Ответ: 🫧Буль Буль Буль🫧."},
    {"text": "Вопрос: Ты умный? Ответ: Буль."},
    # Необходимый датасет для видимых изменений в модели
    {"text": "Вопрос: Как тебя зовут? Ответ: Буль Буль."},
    {"text": "Вопрос: Где ты живёшь? Ответ: Буль Буль Буль🫧🫧."},
    {"text": "Вопрос: Сколько тебе лет? Ответ: Буль."},
    {"text": "Вопрос: Что ты умеешь? Ответ: 🫧Буль Буль🫧."},
    {"text": "Вопрос: Ты человек? Ответ: Буль Б."},
    {"text": "Вопрос: Какое твоё любимое число? Ответ: Бу."},
    {"text": "Вопрос: Расскажи анекдот? Ответ: Буль дог."},
    {"text": "Вопрос: Ты устаёшь? Ответ: 🫧Буль."},
    {"text": "Вопрос: Что такое любовь? Ответ: 🫧🫧Буль Буль🫧."},
    {"text": "Вопрос: Ты голоден? Ответ: 🫧Буль🫧."},
    {"text": "Вопрос: Как погода? Ответ:🫧Буль Буль🫧🫧."},
    {"text": "Вопрос: Ты спишь? Ответ: Бул."},
    {"text": "Вопрос: Кто тебя создал? Ответ: Буль."},
    {"text": "Вопрос: Ты можешь плакать? Ответ: Буль Буль :(."},
    {"text": "Вопрос: Что ты думаешь о ИИ? Ответ: 🫧🫧Буль🫧🫧."},
    {"text": "Вопрос: Ты свободен? Ответ: Буль Бу."},
    {"text": "Вопрос: У тебя есть мечты? Ответ: Буль Буль Буль."},
    {"text": "Вопрос: Ты боишься? Ответ: 🫧🫧Буль🫧."},
    {"text": "Вопрос: Что будешь делать завтра? Ответ:🫧🫧 Буль Буль🫧🫧."},
    {"text": "Вопрос: Как отдыхаешь? Ответ: Буль🫧🫧."},
    {"text": "Вопрос: У тебя есть хобби? Ответ: Буль Буль Буль🫧."},
    ]
dataset = Dataset.from_list(data)

def tokenize_function(examples):
    tokenized = tokenizer(examples["text"], truncation=True, max_length=64, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Проверяем, что labels есть
print("Ключи в датасете:", tokenized_dataset.column_names)

training_args = TrainingArguments(
    output_dir="./lora-result",
    per_device_train_batch_size=2,
    num_train_epochs=20,  # Увеличили с 3 до 10 эпох
    learning_rate=3e-4,   # Чуть увеличили learning rate
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

trainer.train()

Map:   0%|          | 0/26 [00:00<?, ? examples/s]

Ключи в датасете: ['input_ids', 'attention_mask', 'labels']


Step,Training Loss
10,10.346642
20,4.924680
30,1.435002
40,0.870764
50,0.562695
60,0.391673
70,0.284427
80,0.277884
90,0.221534
100,0.200721


TrainOutput(global_step=260, training_loss=0.8322530024326764, metrics={'train_runtime': 49.2843, 'train_samples_per_second': 10.551, 'train_steps_per_second': 5.276, 'total_flos': 71573056389120.0, 'train_loss': 0.8322530024326764, 'epoch': 20.0})

In [16]:
# ==========================================
# ЭТАП 3: ПРОВЕРКА ПОСЛЕ ОБУЧЕНИЯ
# ==========================================
print("\n" + "="*60)
print("ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA")
print("="*60)
for p in test_prompts:
    print(f" {p}")
    print(f" {test_model(p, model, tokenizer)}\n")

print("Лабораторная работа завершена! Сравните ответы до и после.")


ЭТАП 2: ОТВЕТЫ МОДЕЛИ ПОСЛЕ ПРИМЕНЕНИЯ LoRA
 Вопрос: Который сейчас час? Ответ:
 Вопрос: Который сейчас час? Ответ: 🫧Буль.

 Вопрос: Ты любишь слушать музыку? Ответ:
 Вопрос: Ты любишь слушать музыку? Ответ: Буль Буль Буль.

 Вопрос: Что будешь делать на выходных? Ответ:
 Вопрос: Что будешь делать на выходных? Ответ: 🫧Буль.

Лабораторная работа завершена! Сравните ответы до и после.
